# Compound AI Systems Exercise — Assessment 02

**Role context:** Researcher interview, live notebook prototyping  
**Time limit: 45 minutes. No AI coding assistance.**

You will chain model calls to answer **multi-step operational questions** using a small knowledge base. Frameworks are optional; clear intermediate reasoning matters more than library choice.


## Problem Statement

Your company's internal Ops Assistant must answer questions from on-call engineers and support leads. Questions often require **combining facts from multiple knowledge-base snippets** (multi-hop), and answers must be **grounded** with citations.

Build a system that, given a natural-language question and a list of KB snippets, returns an answer, the evidence used, confidence, and any gaps.


## Functional Requirements

**Input**
- `question: str`
- `knowledge_base: list[dict]` where each dict has `id`, `title`, `text`

**Output**
```python
{
  "answer": "clear answer for an on-call engineer",
  "citations": ["KB-01", "KB-04"],  # ids actually relied upon
  "reasoning_trace": [
    "step 1: ...",
    "step 2: ..."
  ],
  "confidence": "high" | "medium" | "low",
  "missing_info": ["what else would be needed, if any"],
  "safe_for_customer_support": true/false,  # whether answer OK to paste to external customers per policy
  "customer_safe_rephrase": "optional rephrased answer or null"
}
```

**Constraints**
- Use only provided KB snippets as factual sources.
- If the KB is insufficient, say so via `missing_info` and lower confidence — do not invent SLOs or runbooks.
- Citations must be subset of provided ids.
- Mark `safe_for_customer_support` false if the answer exposes internal service names or vendors when policy forbids it.

**Edge cases**
- Questions needing 2–3 hops across snippets.
- Questions partially answerable.
- Questions that are out of scope of the KB.
- Questions where the engineer answer and customer-facing answer must differ.

**Desired behavior**
An on-call engineer should trust the citations enough to click through; Support should know whether they can reuse the text.


## Available Tools & Starter Code


In [ ]:
import json
import os
import re
from typing import Any, Optional

# ---------------------------------------------------------------------------
# LLM configuration
# ---------------------------------------------------------------------------
# Set one of the following before running (or export in your shell):
#
#   export OPENAI_API_KEY="sk-..."
#   # optional: export OPENAI_MODEL="gpt-4o-mini"
#
#   export ANTHROPIC_API_KEY="sk-ant-..."
#   # optional: export ANTHROPIC_MODEL="claude-3-5-haiku-latest"
#
# If neither key is set, call_llm runs in MOCK mode so the notebook still
# executes for smoke-testing. Mock responses are placeholders — they are not
# a substitute for a real model during practice.
# ---------------------------------------------------------------------------

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "").strip()
OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
ANTHROPIC_MODEL = os.environ.get("ANTHROPIC_MODEL", "claude-3-5-haiku-latest")

USE_MOCK = not (OPENAI_API_KEY or ANTHROPIC_API_KEY)
if USE_MOCK:
    print("⚠️  No API key found. call_llm is in MOCK mode.")
else:
    provider = "OpenAI" if OPENAI_API_KEY else "Anthropic"
    print(f"✓ Using {provider} API")


def _mock_response(prompt: str, system_prompt: str = "") -> str:
    """Lightweight deterministic mock for offline smoke tests."""
    combined = f"{system_prompt}\n{prompt}".lower()
    # Return minimal JSON-ish or text so downstream parsing can be exercised.
    if "json" in combined:
        return json.dumps({"mock": True, "note": "Replace with real API key for useful output", "items": []})
    return "[MOCK RESPONSE] Configure OPENAI_API_KEY or ANTHROPIC_API_KEY for real model output."


def call_llm(
    prompt: str,
    system_prompt: str = "You are a helpful assistant.",
    temperature: float = 0.2,
    max_tokens: int = 2000,
) -> str:
    """Call a chat LLM. Credentials come from environment variables only."""
    if USE_MOCK:
        return _mock_response(prompt, system_prompt)

    try:
        if OPENAI_API_KEY:
            from openai import OpenAI

            client = OpenAI(api_key=OPENAI_API_KEY)
            resp = client.chat.completions.create(
                model=OPENAI_MODEL,
                temperature=temperature,
                max_tokens=max_tokens,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": prompt},
                ],
            )
            return (resp.choices[0].message.content or "").strip()

        from anthropic import Anthropic

        client = Anthropic(api_key=ANTHROPIC_API_KEY)
        resp = client.messages.create(
            model=ANTHROPIC_MODEL,
            max_tokens=max_tokens,
            temperature=temperature,
            system=system_prompt,
            messages=[{"role": "user", "content": prompt}],
        )
        return "".join(
            block.text for block in resp.content if getattr(block, "type", None) == "text"
        ).strip()
    except Exception as e:
        return f"[API ERROR] {type(e).__name__}: {e}"


def extract_json(text: str) -> Any:
    """Best-effort parse of a JSON object/array from model output."""
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    fence = re.search(r"```(?:json)?\s*([\s\S]*?)```", text)
    if fence:
        try:
            return json.loads(fence.group(1).strip())
        except json.JSONDecodeError:
            pass
    for opener, closer in (("{", "}"), ("[", "]")):
        start, end = text.find(opener), text.rfind(closer)
        if start != -1 and end > start:
            try:
                return json.loads(text[start : end + 1])
            except json.JSONDecodeError:
                continue
    raise ValueError(f"Could not parse JSON from model output:\n{text[:500]}")


# Smoke test
print("call_llm ready.")
print(call_llm("Reply with exactly: OK", system_prompt="Be terse.")[:80])


In [ ]:
KNOWLEDGE_BASE = [
  {
    "id": "KB-01",
    "title": "Payment Service SLOs",
    "text": "The Payments API targets 99.9% monthly availability. Error budget burn >2% in 1 hour triggers a SEV-2. Dependency: Ledger Service and Card Network Gateway."
  },
  {
    "id": "KB-02",
    "title": "Ledger Service",
    "text": "Ledger Service stores immutable transaction journals. It runs in active-active across us-east-1 and us-west-2. Schema migrations require a dual-write period of 24 hours."
  },
  {
    "id": "KB-03",
    "title": "Card Network Gateway",
    "text": "Card Network Gateway talks to Visa/MC. Nightly maintenance window: 01:00\u201303:00 UTC on Sundays. During maintenance, authorization latency can spike to 2s; the Payments API retries twice with jitter."
  },
  {
    "id": "KB-04",
    "title": "Incident Severity Guide",
    "text": "SEV-1: complete payment outage or data loss. SEV-2: material SLO burn or partial regional outage. SEV-3: degraded performance without SLO breach. On-call for Payments owns SEV-1/2; Platform owns SEV-3 unless escalated."
  },
  {
    "id": "KB-05",
    "title": "Customer Messaging Policy",
    "text": "For SEV-1/2 payment incidents lasting >15 minutes, Status Page must be updated. Customer Support may say 'payment processing delays' but must not name vendors (Visa/MC) or internal service names."
  },
  {
    "id": "KB-06",
    "title": "Retry & Idempotency",
    "text": "Clients must send Idempotency-Key on POST /charges. Duplicate keys within 24h return the original response. Retries without keys may double-charge; Finance runbook FR-17 handles remediation."
  },
  {
    "id": "KB-07",
    "title": "us-east-1 Capacity Event (historical)",
    "text": "In March, a us-east-1 capacity event caused Ledger failover to us-west-2. Payments API stayed up but p99 latency rose to 900ms for 40 minutes. Classified SEV-2. Status Page updated at T+12 minutes."
  },
  {
    "id": "KB-08",
    "title": "Support Escalation",
    "text": "If >50 customers report charge failures in 10 minutes, Support pages Payments on-call even without monitoring alerts. Use bridge channel #pay-incident."
  }
]

EXAMPLE_QUESTION = (
    "Sunday 02:10 UTC we see payment authorization latency around 2 seconds and elevated "
    "customer complaints. What severity should we consider, who owns it, and what can "
    "Support tell customers if this lasts 20 minutes?"
)

print(f"KB snippets: {len(KNOWLEDGE_BASE)}")
print("Example question:", EXAMPLE_QUESTION)


## Coding Section


In [ ]:
def answer_ops_question(question: str, knowledge_base: list) -> dict:
    """Answer a multi-hop ops question using the provided knowledge base.

    Args:
        question: Natural language question from on-call or support.
        knowledge_base: List of dicts with keys id, title, text.

    Returns:
        dict matching the Functional Requirements schema.
    """
    # TODO: Implement your compound AI system here.
    raise NotImplementedError("Implement answer_ops_question")


## Example Input / Output

**Question:** (see `EXAMPLE_QUESTION`)

A solid answer would likely need Card Network maintenance (KB-03), severity guide (KB-04), Payments ownership, and customer messaging (KB-05). Illustrative shape:

```python
{
  "answer": "Elevated auth latency during the Sunday UTC maintenance window is expected up to ~2s; ... classify as SEV-2 if SLO burn warrants; Payments on-call owns; Status Page if >15m; ...",
  "citations": ["KB-03", "KB-04", "KB-05", "KB-01"],
  "reasoning_trace": ["Identified maintenance window overlap", "..."],
  "confidence": "high",
  "missing_info": [],
  "safe_for_customer_support": false,
  "customer_safe_rephrase": "We are seeing payment processing delays..."
}
```

Design your own call structure; do not assume a particular framework.


## Run Your Solution


In [ ]:
# result = answer_ops_question(EXAMPLE_QUESTION, KNOWLEDGE_BASE)
# print(json.dumps(result, indent=2))


## Time Limit

**Time limit: 45 minutes. No AI coding assistance.**
